In [4]:
import pandas as pd

df = pd.read_csv("sales result.csv")



print("========== ORIGINAL DATA ==========")
print(df.head())
print("\nShape:", df.shape)



# 2. BEFORE CLEANING - DATA QUALITY CHECK


print("\n========== DATA INFORMATION ==========")
df.info()

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\n========== DUPLICATE ROWS ==========")
print("Duplicates:", df.duplicated().sum())


# 3. SAVE BEFORE VALUES FOR LOG


rows_before = len(df)
columns_before = len(df.columns)
missing_before = df.isnull().sum().sum()
duplicates_before = df.duplicated().sum()


# 4. HANDLE MISSING VALUES


# Numeric columns
if "quantity" in df.columns:
    df["quantity"] = df["quantity"].fillna(
        df["quantity"].median()
    )

if "price" in df.columns:
    df["price"] = df["price"].fillna(
        df["price"].mean()
    )


# Categorical columns
if "city" in df.columns:
    if df["city"].isnull().sum() > 0:
        df["city"] = df["city"].fillna(
            df["city"].mode()[0]
        )

if "category" in df.columns:
    df["category"] = df["category"].ffill()



# 5. REMOVE DUPLICATES


df = df.drop_duplicates()



# 6. OUTLIER DETECTION USING IQR


outliers_before = 0

if "price" in df.columns:

    Q1 = df["price"].quantile(0.25)
    Q3 = df["price"].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Count outliers
    outlier_condition = (
        (df["price"] < lower_bound)
        | (df["price"] > upper_bound)
    )

    outliers_before = outlier_condition.sum()

    print("\n========== PRICE OUTLIERS ==========")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outliers detected:", outliers_before)

    # Cap outliers
    df["price"] = df["price"].clip(
        lower=lower_bound,
        upper=upper_bound
    )



# 7. STANDARDISE STRING VALUES


if "customer_name" in df.columns:
    df["customer_name"] = (
        df["customer_name"]
        .astype("string")
        .str.strip()
    )

if "product" in df.columns:
    df["product"] = (
        df["product"]
        .astype("string")
        .str.strip()
        .str.title()
    )

if "category" in df.columns:
    df["category"] = (
        df["category"]
        .astype("string")
        .str.strip()
        .str.title()
    )

if "city" in df.columns:
    df["city"] = (
        df["city"]
        .astype("string")
        .str.strip()
        .str.title()
    )



# 8. AFTER CLEANING CHECK


missing_after = df.isnull().sum().sum()
duplicates_after = df.duplicated().sum()
rows_after = len(df)

columns_after = len(df.columns)


print("\n========== AFTER CLEANING ==========")

print("\nMissing values:")
print(df.isnull().sum())

print("\nTotal missing values:", missing_after)
print("Duplicates:", duplicates_after)
print("Rows:", rows_after)
print("Columns:", columns_after)



# 9. DISPLAY CLEANED DATA


print("\n========== CLEANED DATA ==========")
print(df.head(10))



# 10. EXPORT CLEANED DATASET


df.to_csv("cleaned_dataset.csv", index=False)

print("\n====================================")
print("Cleaning completed successfully!")
print("File created: cleaned_dataset.csv")


========== ORIGINAL DATA ==========
   sale_id customer_name product     category  quantity    price       city
0        1         Rahul  Laptop  Electronics         1  55000.0  Bangalore
1        2         Priya   Phone  Electronics         2  25000.0     Mumbai
2        3          Arun   Chair    Furniture         4   3500.0  Bangalore
3        4         Sneha  Laptop  Electronics         1  60000.0      Delhi
4        5         Kiran   Table    Furniture         2   7000.0    Chennai

Shape: (10, 7)

========== DATA INFORMATION ==========
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_id        10 non-null     int64  
 1   customer_name  10 non-null     str    
 2   product        10 non-null     str    
 3   category       10 non-null     str    
 4   quantity       10 non-null     int64  
 5   price          10 non-null     float64
 6   city 